# Multi-Hop Retrieval — Train QueryEncoder (Model 2, retrieval_v3)

**Before running:**
1. Settings → Accelerator → **T4 GPU**
2. Settings → Internet → **On**
3. Google Drive `musique_data/` folder must contain:
   - `musique_ans_v1.0_train.jsonl`, `musique_ans_v1.0_dev.jsonl`
   - `generator_best.pt`  ← from train_generator_kaggle.ipynb (REQUIRED)

**What this trains:**
- `QueryEncoder` (Model 2) — BERT → mean-pool → proj → 128-dim L2-norm
- Warm-started from `generator_best.pt` (encoder1 + proj weights)
- **Generator (Model 1) is FROZEN** — only QueryEncoder trains
- **Loss:** MarginRankingLoss — align Q with G(A,B_pos) vs G(A,B_neg)
- This is what makes FULL eval a FAIR test: maps queries into the complement space

**Expected time: ~1.5 hours on T4**

**After training:** download `model2_best.pt` → upload to Drive `musique_data/`

In [ ]:
# ── [EDIT IF NEEDED] ─────────────────────────────────────────────────
REPO_URL        = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME       = "multihop-retrieval"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # musique_data folder ID
WORK_DIR        = f"/kaggle/working/{REPO_NAME}/retrieval_v3"
# ─────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU — must be T4
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → T4 GPU")
cc = torch.cuda.get_device_properties(0).major
if cc < 7:
    raise RuntimeError("GPU is P100 (sm_60) — change to T4 GPU")
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os
repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")
os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())
os.system("pip install -q 'transformers>=4.35.0' 'rank_bm25>=0.2.2' gdown")
print("Dependencies ready")

In [ ]:
# 3. Download data + generator checkpoint from Google Drive
import os, gdown
download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=download_dir, quiet=False, use_cookies=False)
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    print(f"  {f:45s}  {os.path.getsize(f'{download_dir}/{f}')/1e6:7.1f} MB")

m1_path = f"{download_dir}/generator_best.pt"
if not os.path.exists(m1_path):
    raise FileNotFoundError(
        "generator_best.pt not found in Drive folder.\n"
        "Run train_generator_kaggle.ipynb first and upload generator_best.pt to musique_data/"
    )
print(f"\ngenerator_best.pt: {os.path.getsize(m1_path)/1e6:.0f} MB  OK")

In [ ]:
# 4. Set up file paths — symlink data into retrieval_v2/ (data_loader lives there)
import os, shutil
drive   = "/kaggle/working/drive_data"
v2_data = f"/kaggle/working/{REPO_NAME}/retrieval_v2/data/musique"

os.makedirs(v2_data,              exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache",  exist_ok=True)

for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src, dst = f"{drive}/{fname}", f"{v2_data}/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"  {fname}: OK")

# Copy generator_best.pt into v3 models/
m1_dst = f"{WORK_DIR}/models/generator_best.pt"
if not os.path.exists(m1_dst):
    print("  Copying generator_best.pt...", flush=True)
    shutil.copy(f"{drive}/generator_best.pt", m1_dst)
print(f"  generator_best.pt: OK ({os.path.getsize(m1_dst)/1e6:.0f} MB)")
print("\nAll paths ready")

In [ ]:
# 5. Smoke test — verify data loader, generator loading, QueryEncoder init
import os
os.chdir(WORK_DIR)
print("=== Smoke test (50 examples) ===")
os.system("python model2_train.py --smoke")

---
## Train QueryEncoder — Full 3-epoch run

Per training step:
1. `comp_pos = G.extract_complement(A, B_pos)` — frozen, no grad
2. `comp_neg_i = G.extract_complement(A, B_neg_i)` × 3 — frozen, no grad
3. `q_vec = QueryEncoder(Q)` — with grad
4. MarginRankingLoss: `dot(q_vec, comp_pos) > dot(q_vec, comp_neg_i) + margin`

In [ ]:
# 6. Train QueryEncoder (full run)
import os
os.chdir(WORK_DIR)
log_file = "/kaggle/working/model2_train.log"
os.system(f"python model2_train.py 2>&1 | tee {log_file}")
print("\nTraining complete")

In [ ]:
# 7. Verify + copy model2_best.pt to /kaggle/working/ for download
import os, shutil
best = f"{WORK_DIR}/models/model2_best.pt"
if os.path.exists(best):
    out_path = "/kaggle/working/model2_best.pt"
    shutil.copy(best, out_path)
    print(f"  model2_best.pt: {os.path.getsize(out_path)/1e6:.1f} MB  OK")
    print(f"  Copied to {out_path}")
    print("  → Download from Output tab → upload to Drive musique_data/")
else:
    print("  model2_best.pt NOT FOUND — check training log")

print("\n--- Last 20 lines of training log ---")
with open("/kaggle/working/model2_train.log") as f:
    print("".join(f.readlines()[-20:]))

---
## Done

`model2_best.pt` is in the **Output tab**.

1. Download it → upload to Drive `musique_data/`
2. Run `eval_kaggle.ipynb` — now FULL uses Model 2 query encoding (fair test)